# Lab 01 — Security Data Exploration

## Research question
What information does the UNSW-NB15 prepared dataset contain, and what properties could affect a machine-learning threat detector?

## Learning objectives
- inspect the dataset before modeling;
- understand `label` and `attack_cat`;
- identify numeric and categorical features;
- measure class balance;
- detect possible leakage and data-quality concerns.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_unsw, load_feature_dictionary

train, test = load_unsw(DATA_DIR)
feature_dictionary = load_feature_dictionary(DATA_DIR)

print("Training shape:", train.shape)
print("Testing shape :", test.shape)
display(train.head())

## Understand the targets

`label` supports **binary classification**:

- `0` = normal
- `1` = attack

`attack_cat` supports **multiclass classification** by attack category.

Neither target should be used as an input feature when predicting the other.

In [ ]:
display(train["label"].value_counts(dropna=False).rename("count").to_frame())
display(train["attack_cat"].value_counts(dropna=False).rename("count").to_frame())

In [ ]:
ax = train["label"].value_counts().sort_index().plot(
    kind="bar",
    figsize=(7, 4)
)
ax.set_title("Binary class distribution")
ax.set_xlabel("0 = normal, 1 = attack")
ax.set_ylabel("Count")
plt.show()

In [ ]:
numeric_cols = train.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_cols = [c for c in train.columns if c not in numeric_cols]

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))
print(categorical_cols)

missing = train.isna().mean().sort_values(ascending=False)
display(missing[missing > 0].head(20).rename("missing_fraction").to_frame())

## Your analysis

Answer in a Markdown cell:

1. Is the binary target balanced?
2. Which attack categories are rare?
3. Which features are categorical?
4. Is there an identifier column that should not drive prediction?
5. What could create target leakage?
6. What does the model *not* know from this dataset?